In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("apartments_data-1.csv")

In [4]:
import os
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

In [5]:
import torch
import json
import re
from transformers import pipeline

model_name = "Qwen/Qwen2.5-1.5B-Instruct"
if torch.cuda.is_available():
    device = "cuda"
    model_kwargs = {"torch_dtype": torch.float16}
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
    model_kwargs = {"torch_dtype": torch.float16}
else:
    device = "cpu"
    model_kwargs = {"torch_dtype": torch.float32}

print(f"Loading {model_name} on {device}...")
generator = pipeline(
    "text-generation",
    model=model_name,
    device=device,
    **model_kwargs
)
print("Model loaded.")

Loading Qwen/Qwen2.5-1.5B-Instruct on mps...


`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use mps


Model loaded.


In [9]:

def chat1(query):
    prompt = f"""You are an apartment broker assistant. Read the following user query and extract apartment preferences.
    Return ONLY a JSON object with these exact keys. If a value is not mentioned, use null.
    Keys to extract:
    - neighborhood (string)
    - rent (number, monthly USD)
    - bedrooms (number)
    - bathrooms (number)
    - square_feet (number)
    - floor (number)
    - in_unit_laundry (boolean or null)
    - dishwasher (boolean or null)
    - central_ac (boolean or null)
    - parking (boolean or null)
    - gym (boolean or null)
    - balcony (boolean or null)
    - pet (boolean or null)
    - heat_included (boolean or null)
    - water_included (boolean or null)

    Rules:
    - Return ONLY valid JSON. No explanation, no extra text.
    - Do not invent or infer values not explicitly stated.
    - Use null for anything not mentioned.

    Query: {query}
    """
    messages = [{"role": "user", "content": prompt}]
    response = generator(messages, max_new_tokens=500, do_sample=False)
    raw = response[0]['generated_text'][-1]['content']
    print(raw)
    return raw




In [11]:
# --- Example usage ---
if __name__ == "__main__":
    queries = [
        "I am looking for a 2 bedroom apartment in Jamaica Plain under $2000 a month with in-unit laundry. I want to be within a 2 mile radius to a gym and I want updated appliances"
    ]

    for q in queries:
        print(f"Query: {q}")
        print(chat1(q))
        

Query: I am looking for a 2 bedroom apartment in Jamaica Plain under $2000 a month with in-unit laundry. I want to be within a 2 mile radius to a gym and I want updated appliances
```json
{
  "neighborhood": "Jamaica Plain",
  "rent": 1975,
  "bedrooms": 2,
  "bathrooms": null,
  "square_feet": null,
  "floor": null,
  "in_unit_laundry": true,
  "dishwasher": null,
  "central_ac": null,
  "parking": null,
  "gym": true,
  "balcony": null,
  "pet": null,
  "heat_included": null,
  "water_included": null
}
```
```json
{
  "neighborhood": "Jamaica Plain",
  "rent": 1975,
  "bedrooms": 2,
  "bathrooms": null,
  "square_feet": null,
  "floor": null,
  "in_unit_laundry": true,
  "dishwasher": null,
  "central_ac": null,
  "parking": null,
  "gym": true,
  "balcony": null,
  "pet": null,
  "heat_included": null,
  "water_included": null
}
```
